In [16]:
import pandas as pd  
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers, models
from tensorflow.keras import regularizers
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import os

# Path to the NEW polar dataset
dataset_path = r'C:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Dataset_Creator\poses_dataset_polar.csv'
df = pd.read_csv(dataset_path)

In [17]:
# 1. Data Preparation for LSTM (Polar Features)
# Selecionamos as colunas de features polares e normalizadas geradas pelo main.py
feature_cols = [
    'angle_knee_l', 'angle_knee_r', 'angle_hip_l', 'angle_hip_r',
    'dist_hip_ankle_l', 'dist_hip_ankle_r'
]
X_advanced = df[feature_cols].values
y_raw = df['label'].values

# --- SLIDING WINDOW FOR LSTM ---
# Para LSTM, X deve ter a forma: (amostras, timesteps, features)
WINDOW_SIZE = 30
X_windows, y_windows = [], []

for video in df['video_name'].unique():
    mask = df['video_name'] == video
    video_data = X_advanced[mask]
    video_labels = y_raw[mask]
    
    for i in range(0, len(video_data) - WINDOW_SIZE):
        window = video_data[i : i + WINDOW_SIZE] # Forma: (30, 6)
        X_windows.append(window)
        y_windows.append(1 if np.any(video_labels[i : i + WINDOW_SIZE] == 1) else 0)

X = np.array(X_windows)
y = np.array(y_windows)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
print(f"Final Dataset Shape for LSTM: {X.shape}") # Esperado: (amostras, 30, 6)

Final Dataset Shape for LSTM: (11692, 30, 6)


In [18]:
# Escalonamento para dados 3D (LSTM)
scaler = StandardScaler()
# Achata para escalar as features individualmente: (amostras * 30, 6)
X_train_reshaped = X_train.reshape(-1, X_train.shape[-1])
scaler.fit(X_train_reshaped)

# Aplica o escalonamento e retorna para a forma original (amostras, 30, 6)
X_train = scaler.transform(X_train_reshaped).reshape(X_train.shape)
X_test_reshaped = X_test.reshape(-1, X_test.shape[-1])
X_test = scaler.transform(X_test_reshaped).reshape(X_test.shape)
X_val_reshaped = X_val.reshape(-1, X_val.shape[-1])
X_val = scaler.transform(X_val_reshaped).reshape(X_val.shape)

In [19]:
def build_kick_model(input_shape):
    """
    Arquitetura aprimorada: Bidirectional LSTM com Regularização Forte.
    Ajuda a reduzir falsos positivos por distância e melhora a detecção de ambos os lados.
    """
    model = models.Sequential([
        # Bidirectional LSTM: Analisa a sequência nos dois sentidos (passado e futuro)
        layers.Bidirectional(layers.LSTM(64, return_sequences=True), input_shape=input_shape),
        layers.Dropout(0.3),
        
        # Segunda camada para extrair características complexas
        layers.Bidirectional(layers.LSTM(32, return_sequences=False)),
        layers.Dropout(0.3),

        # Camada Densa com Regularização L2 para evitar overfitting em distâncias
        layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
        layers.Dense(1, activation='sigmoid')
    ])

    # Optimizer com learning rate menor para convergência mais suave
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
    
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [20]:
early_stop = EarlyStopping(
      monitor='val_loss',
      patience=10,          # Wait 10 epochs before giving up
      restore_best_weights=True # Give me the model from the best epoch, not the last one
  )

In [21]:
# Treinamento do modelo LSTM
# O input_shape agora é (30, 6)
nn_model = build_kick_model((X_train.shape[1], X_train.shape[2]))

nn_model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val), callbacks=[early_stop])

print(nn_model.evaluate(X_test, y_test))

Epoch 1/100


c:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\.venv\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


234/234 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.7855 - loss: 0.6324 - val_accuracy: 0.8258 - val_loss: 0.4997
Epoch 2/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8430 - loss: 0.4543 - val_accuracy: 0.8482 - val_loss: 0.4002
Epoch 3/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8710 - loss: 0.3726 - val_accuracy: 0.8808 - val_loss: 0.3399
Epoch 4/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.8880 - loss: 0.3215 - val_accuracy: 0.8985 - val_loss: 0.2954
Epoch 5/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.9072 - loss: 0.2873 - val_accuracy: 0.9113 - val_loss: 0.2643
Epoch 6/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.9167 - loss: 0.2517 - val_accuracy: 0.9262 - val_loss: 0.2268
Epoch 7/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.9245 - loss: 0.2317 - val_accuracy: 0.9156 - val_loss: 0.2459
Epoch 8/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.9289 - loss: 0.2153 - val_accurac

In [22]:
def evaluate_model(name, model, X_test, y_test):
      # For NN, we need to convert probabilities to 0 or 1
      if hasattr(model, 'predict_proba') or 'RandomForest' in str(type(model)):
          preds = model.predict(X_test)
      else: # TensorFlow model
          probs = model.predict(X_test)
          preds = (probs > 0.5).astype(int).flatten()

      print(f"\n--- {name} Report ---")
      print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
      print(classification_report(y_test, preds))

In [25]:
evaluate_model("Neural Network", nn_model, X_test, y_test)

74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

--- Neural Network Report ---
Accuracy: 0.9880
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1729
           1       0.99      0.96      0.98       610

    accuracy                           0.99      2339
   macro avg       0.99      0.98      0.98      2339
weighted avg       0.99      0.99      0.99      2339



In [24]:
# Save model and scaler to the correct folder
model_save_path = r'C:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Pose_Predict\model\kick_detection_model.h5'
scaler_save_path = r'C:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Pose_Predict\model\scaler.pkl'

nn_model.save(model_save_path)
joblib.dump(scaler, scaler_save_path)
print(f"Model saved to {model_save_path}")
print(f"Scaler saved to {scaler_save_path}")

Model saved to C:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Pose_Predict\model\kick_detection_model.h5
Scaler saved to C:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Pose_Predict\model\scaler.pkl
